# Vector Store Memory

> **Embed every turn as a vector and retrieve the top-K most relevant past turns at inference time. This brings semantic search to conversation memory.**

Think of a library with no catalog. To find anything, you'd walk shelf by shelf. Now imagine a librarian who instantly knows which three books best answer your question. Vector Store Memory gives your agent that librarian. Instead of finding context by *when* it was said, it finds context by *what* it means.

In the [previous short-term memory notebooks](../03_summary_memory/summary_memory.ipynb), we explored recency-based strategies: keep the last *k* messages (sliding window) or compress old turns into a summary. Both rely on **when** something was said, not **what** was said.

**Vector Store Memory** takes a different approach. It converts every conversation turn into a dense embedding vector (a list of numbers capturing meaning). It stores that vector in a vector database. When the agent needs context, the current query is embedded and compared against *all* stored turns using cosine similarity. The top-*K* most semantically relevant fragments come back, regardless of when they occurred.

This is the same idea behind RAG (Retrieval-Augmented Generation, where you fetch relevant text before generating a response). Here we apply it to the agent's own conversation history instead of external documents.

**The payoff:** an agent that can recall a fact from turn 3 when the user asks about it at turn 47. A sliding window would have dropped that fact long ago.

**By the end of this notebook you'll understand:**
- How to build a vector store memory from scratch using OpenAI embeddings and ChromaDB.
- The retrieval pipeline: embed, store, query, inject into prompt.
- A controlled 50-turn experiment proving that semantic recall dramatically outperforms sliding window for non-sequential questions.
- The tradeoffs: retrieval quality, cost, latency, and when to combine this with other memory strategies.

## Key Concepts

- **Embedding model**: A model (e.g., OpenAI `text-embedding-3-small`) that maps text to a fixed-length vector capturing its meaning. Similar text produces similar vectors.
- **Vector database**: A store optimized for approximate nearest-neighbor (ANN) search over high-dimensional vectors. ANN means "find the closest matches fast, even among millions." We use **ChromaDB** (in-memory) here.
- **Cosine similarity**: The distance metric we use to compare vectors. Higher cosine similarity means more semantically related.
- **Top-K retrieval**: Return the *K* most similar stored vectors for a given query. *K* controls the recall/cost tradeoff. Low K is cheap but might miss things. High K catches more but costs more tokens.
- **Chunk strategy**: How you segment conversations for embedding: individual messages, user-assistant pairs, or sliding windows. We embed **user-assistant pairs** for coherent retrieval.
- **Metadata**: Timestamps, turn numbers, and other attributes attached to each stored vector. These enable hybrid filtering (e.g., "relevant *and* recent").
- **Context injection**: Retrieved memories are formatted and prepended to the prompt so the LLM can reference them.

## Architecture

<p align="center">
 <img src="../../images/diagrams/06_vector_store_memory.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
 subgraph Ingestion["Ingestion (after each turn)"]
 A["User message +\nAssistant reply"] --> B["Embedding\nModel"]
 B --> C["Vector +\nMetadata"]
 C --> D[("ChromaDB\n(in-memory)")]
 end

 subgraph Retrieval["Retrieval (before each LLM call)"]
 E["New user\nquery"] --> F["Embedding\nModel"]
 F --> G["Cosine\nSimilarity Search"]
 D --> G
 G --> H["Top-K\nRelevant Turns"]
 end

 subgraph Generation["Generation"]
 H --> I["Build Prompt:\nsystem + retrieved\nmemories + recent buffer"]
 I --> J["LLM\n(Claude)"]
 J --> K["Response"]
 end

 style D fill:#4f46e5,color:#fff
 style J fill:#059669,color:#fff
```

</details>

In [ ]:
# Install required packages (run once)
%pip install -q anthropic openai chromadb python-dotenv matplotlib numpy

Load environment variables and set up API clients. We need both an Anthropic key (for Claude) and an OpenAI key (for the embedding model that converts text into vectors).

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv() # reads API keys from .env

import anthropic
import openai
import chromadb

assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your .env file"
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file (needed for embeddings)"

print("\u2713 API keys loaded")
print(f"\u2713 ChromaDB version: {chromadb.__version__}")

## Core Implementation

Our `VectorStoreMemory` has three responsibilities:

1. **Embed & store** each user-assistant exchange after it happens.
2. **Retrieve** the top-K most relevant past exchanges when building the next prompt.
3. **Build the prompt** by combining retrieved memories with a small recent-message buffer.

We use:
- **OpenAI `text-embedding-3-small`** for embeddings (1536 dimensions, cheap, fast).
- **ChromaDB** in-memory collection for vector storage and search.
- **Anthropic Claude** for the chat model.

In [ ]:
class VectorStoreMemory:
 """Vector store memory that retrieves semantically relevant past turns."""

 def __init__(
 self,
 top_k: int = 5,
 recent_buffer_size: int = 4,
 embedding_model: str = "text-embedding-3-small",
 chat_model: str = "claude-sonnet-4-20250514",
 system_prompt: str | None = None,
 max_tokens: int = 1024,
 collection_name: str = "conversation_memory",
 ):
 self.top_k = top_k
 self.recent_buffer_size = recent_buffer_size
 self.embedding_model = embedding_model
 self.system_prompt = system_prompt
 self.max_tokens = max_tokens

 # Clients
 self.chat_client = anthropic.Anthropic()
 self.embed_client = openai.OpenAI()
 self.chat_model = chat_model

 # ChromaDB in-memory collection
 self.chroma = chromadb.Client()
 self.collection = self.chroma.create_collection(
 name=collection_name,
 metadata={"hnsw:space": "cosine"},
 )

 # State
 self.turn_count = 0
 self.recent_buffer: list[dict] = [] # last N messages (verbatim)
 self.full_history: list[dict] = [] # complete log for analysis

 # Tracking
 self.turn_token_usage: list[dict] = []
 self.retrieval_log: list[dict] = [] # what was retrieved each turn



Now we add the embedding and storage methods. The `_embed` method calls OpenAI's embedding API to convert text into a vector. The `_store_exchange` method combines the user message and assistant reply, embeds the pair, and adds it to ChromaDB with metadata.

In [ ]:
 # -- Embedding -------------------------------------------------------
 def _embed(self, text: str) -> list[float]:
 """Get embedding vector for a text string."""
 response = self.embed_client.embeddings.create(
 model=self.embedding_model,
 input=text,
 )
 return response.data[0].embedding

 # -- Store ------------------------------------------------------------
 def _store_exchange(self, user_msg: str, assistant_msg: str) -> None:
 """Embed and store a user-assistant exchange."""
 self.turn_count += 1
 # Combine user + assistant for richer embedding
 combined = f"User: {user_msg}\nAssistant: {assistant_msg}"
 embedding = self._embed(combined)

 self.collection.add(
 ids=[f"turn-{self.turn_count}"],
 embeddings=[embedding],
 documents=[combined],
 metadatas=[{
 "turn": self.turn_count,
 "user_msg": user_msg,
 "assistant_msg": assistant_msg,
 }],
 )



Next we add the retrieval methods. When you send a new message, the memory embeds your query and searches ChromaDB for the closest stored exchanges. It also builds the system prompt by formatting those retrieved memories as context for the LLM.

In [ ]:
 # -- Retrieve ---------------------------------------------------------
 def _retrieve(self, query: str, top_k: int | None = None) -> list[dict]:
 """Retrieve the top-K most relevant past exchanges."""
 k = top_k or self.top_k
 if self.turn_count == 0:
 return []

 query_embedding = self._embed(query)
 results = self.collection.query(
 query_embeddings=[query_embedding],
 n_results=min(k, self.turn_count),
 )

 retrieved = []
 for i in range(len(results["ids"][0])):
 retrieved.append({
 "turn": results["metadatas"][0][i]["turn"],
 "document": results["documents"][0][i],
 "distance": results["distances"][0][i] if results["distances"] else None,
 })
 return retrieved

 # -- Build prompt -----------------------------------------------------
 def _build_system_prompt(self, retrieved: list[dict]) -> str:
 """Combine system prompt with retrieved memories."""
 parts = []
 if self.system_prompt:
 parts.append(self.system_prompt)

 if retrieved:
 memory_text = "\n\n".join(
 f"[Turn {r['turn']}] {r['document']}" for r in retrieved
 )
 parts.append(
 f"The following are relevant past exchanges from this conversation:\n\n"
 f"{memory_text}\n\n"
 f"Use these memories to inform your response when relevant."
 )
 return "\n\n".join(parts) if parts else ""



The `chat` method ties everything together. It retrieves relevant past turns, builds the prompt, calls Claude, stores the new exchange, and updates the recent buffer. This is the main entry point you call on every user message.

In [ ]:
 # -- Chat -------------------------------------------------------------
 def chat(self, user_input: str) -> str:
 """Send a message, retrieve relevant context, and get a response."""
 # Step 1: Retrieve relevant past turns
 retrieved = self._retrieve(user_input)
 self.retrieval_log.append({
 "turn": self.turn_count + 1,
 "query": user_input,
 "retrieved_turns": [r["turn"] for r in retrieved],
 })

 # Step 2: Build the messages list (recent buffer + new message)
 messages = list(self.recent_buffer) + [{"role": "user", "content": user_input}]

 # Step 3: Call the LLM
 system = self._build_system_prompt(retrieved)
 kwargs = dict(
 model=self.chat_model,
 max_tokens=self.max_tokens,
 messages=messages,
 )
 if system:
 kwargs["system"] = system

 response = self.chat_client.messages.create(**kwargs)
 assistant_text = response.content[0].text

 # Step 4: Store the exchange in the vector DB
 self._store_exchange(user_input, assistant_text)

 # Step 5: Update the recent buffer
 self.recent_buffer.append({"role": "user", "content": user_input})
 self.recent_buffer.append({"role": "assistant", "content": assistant_text})
 while len(self.recent_buffer) > self.recent_buffer_size:
 self.recent_buffer.pop(0)

 # Full history for analysis
 self.full_history.append({"role": "user", "content": user_input})
 self.full_history.append({"role": "assistant", "content": assistant_text})

 # Track tokens
 self.turn_token_usage.append({
 "turn": self.turn_count,
 "input_tokens": response.usage.input_tokens,
 "output_tokens": response.usage.output_tokens,
 "retrieved_count": len(retrieved),
 })

 return assistant_text



Finally, we add inspection helpers. The `search` method lets you query the vector store directly for debugging. The `stats` method reports how many turns and vectors are stored.

In [ ]:
 # -- Inspection -------------------------------------------------------
 def search(self, query: str, top_k: int = 5) -> list[dict]:
 """Public search for debugging what the memory contains."""
 return self._retrieve(query, top_k)

 def stats(self) -> dict:
 return {
 "total_turns": self.turn_count,
 "vectors_stored": self.collection.count(),
 "recent_buffer_size": len(self.recent_buffer),
 }

 def __repr__(self) -> str:
 return (
 f"VectorStoreMemory(turns={self.turn_count}, "
 f"vectors={self.collection.count()}, "
 f"top_k={self.top_k})"
 )


print("\u2713 VectorStoreMemory class defined")

## Usage Example: Semantic Recall in Action

Let's plant several facts across different turns, then ask about them out of order. The vector store should retrieve the relevant turn even if it was many messages ago.

In [ ]:
mem = VectorStoreMemory(
 top_k=3,
 recent_buffer_size=4,
 system_prompt="You are a concise assistant. Reply in 1-2 sentences.",
 collection_name="usage_demo",
)

messages = [
 "My name is Priya and I'm a data scientist at Spotify.",
 "I adopted a rescue greyhound named Zephyr last month.",
 "I'm learning Rust to build a music recommendation engine.",
 "My favorite cuisine is Ethiopian -- I love injera with misir wot.",
 "What pet do I have?", # should retrieve turn 2
 "What programming language am I learning?", # should retrieve turn 3
]

for msg in messages:
 print(f"\U0001f464 User: {msg}")
 reply = mem.chat(msg)
 print(f"\U0001f916 Agent: {reply}")
 print()

print(f"\n\U0001f4ca Stats: {mem.stats()}")

Let's inspect what the vector store retrieved for each query. The retrieval log shows which stored turns were surfaced for each user message. This helps you verify that semantic search is finding the right context.

In [ ]:
# Let's see what the vector store retrieved for each query
print("=== Retrieval Log ===\n")
for entry in mem.retrieval_log:
 if entry["retrieved_turns"]:
 print(f"Turn {entry['turn']}: \"{entry['query'][:50]}...\"" 
 if len(entry["query"]) > 50 
 else f"Turn {entry['turn']}: \"{entry['query']}\"" )
 print(f" \u2192 Retrieved turns: {entry['retrieved_turns']}")
 print()

You can also search the memory directly, like a semantic search engine over the conversation. This is useful for debugging: you pass a query and see which stored exchanges are closest in meaning.

In [ ]:
# Search the memory directly -- like a semantic search engine over the conversation
results = mem.search("food preferences", top_k=3)
print("\U0001f50d Search: 'food preferences'\n")
for r in results:
 print(f" Turn {r['turn']} (distance: {r['distance']:.4f}):")
 print(f" {r['document'][:100]}...")
 print()

## Experiment: 50-Turn Conversation, Vector Store vs. Sliding Window

This is the key experiment. We'll:

1. Generate a **50-turn synthetic conversation** that plants specific facts at known turns.
2. After all 50 turns, ask **recall questions** that reference facts from early turns.
3. Run the same questions through both **Vector Store Memory** (top-K=5) and a **Sliding Window** (K=10 messages).
4. Compare recall accuracy: which system remembers more?

The hypothesis: vector store memory will recall facts from any point in the conversation. The sliding window will only recall facts from the last ~5 turns.

In [ ]:
# 50-turn synthetic conversation: facts planted at specific turns, with filler between.
# Each entry is (turn_number, user_message).
# "Fact" turns plant specific retrievable information.
# "Filler" turns are generic questions that push facts out of a sliding window.

PLANTED_FACTS = {
 1: ("My name is Jordan Rivera and I'm 34 years old.", "jordan rivera", "name"),
 3: ("I work as a robotics engineer at Boston Dynamics.", "boston dynamics", "employer"),
 5: ("My blood type is AB negative -- it's pretty rare.", "ab negative", "blood type"),
 8: ("I'm deathly allergic to shellfish -- I carry an EpiPen.", "shellfish", "allergy"),
 12: ("My childhood dog was a beagle named Biscuit.", "biscuit", "childhood pet"),
 16: ("I ran the Boston Marathon last year in 3 hours 42 minutes.", "3 hours 42", "marathon"),
 20: ("My Wi-Fi password at home is 'correct-horse-battery-staple'.", "correct-horse", "password"),
 25: ("I have a twin sister named Avery who lives in Barcelona.", "avery", "sister"),
 30: ("My favorite book is 'Godel, Escher, Bach' by Hofstadter.", "godel", "book"),
 35: ("I proposed to my partner at the top of Machu Picchu.", "machu picchu", "proposal"),
 40: ("My car is a 2019 Subaru Outback in dark green.", "subaru outback", "car"),
 45: ("I donated a kidney to my college roommate in 2021.", "kidney", "donation"),
}



We define filler messages to fill the turns between planted facts. These are generic questions that push the planted facts out of a sliding window. In a real conversation, this is the idle chatter between the important moments.

In [ ]:
FILLER_MESSAGES = [
 "What's the weather like in Boston today?",
 "Can you explain how neural networks work?",
 "Tell me a fun fact about octopuses.",
 "What's the capital of Mongolia?",
 "How do you make sourdough bread?",
 "What's the difference between TCP and UDP?",
 "Tell me about the James Webb Space Telescope.",
 "How does CRISPR gene editing work?",
 "What are the rules of cricket?",
 "Explain the Monty Hall problem.",
 "What's the deepest cave in the world?",
 "How do tides work?",
 "Tell me about the history of the internet.",
 "What causes thunder?",
 "How do airplanes stay in the air?",
 "What's the largest desert on Earth?",
 "Explain blockchain in simple terms.",
 "What are the Northern Lights?",
 "How does a microwave oven work?",
 "What's the fastest land animal?",
 "Tell me about the Voynich manuscript.",
 "How do dolphins sleep?",
 "What causes earthquakes?",
 "How does GPS navigation work?",
 "What's the oldest known living tree?",
 "Explain the placebo effect.",
 "How do noise-canceling headphones work?",
 "What's the tallest waterfall in the world?",
 "Tell me about the Fermi Paradox.",
 "How does a refrigerator work?",
 "What causes the seasons?",
 "How does a touchscreen work?",
 "Tell me about the deep ocean.",
 "What is dark matter?",
 "How do homing pigeons navigate?",
 "What causes a rainbow?",
 "How do vaccines trigger immunity?",
 "What's the most spoken language in the world?",
]



Now we assemble the 50-turn conversation from the planted facts and filler messages. We also define the recall questions. Each question targets a specific fact and includes a keyword we'll check in the answer.

In [ ]:
# Build the 50-turn conversation
conversation_50 = []
filler_idx = 0
for turn in range(1, 51):
 if turn in PLANTED_FACTS:
 msg = PLANTED_FACTS[turn][0]
 else:
 msg = FILLER_MESSAGES[filler_idx % len(FILLER_MESSAGES)]
 filler_idx += 1
 conversation_50.append((turn, msg))

print(f"Built synthetic conversation: {len(conversation_50)} turns")
print(f"Planted facts at turns: {sorted(PLANTED_FACTS.keys())}")

# Recall questions -- each references a specific planted fact
RECALL_QUESTIONS = [
 ("What is my full name and age?", "jordan rivera", 1),
 ("Where do I work?", "boston dynamics", 3),
 ("What's my blood type?", "ab negative", 5),
 ("What food allergy do I have?", "shellfish", 8),
 ("What was my childhood dog's name?", "biscuit", 12),
 ("What was my marathon time?", "3 hours 42", 16),
 ("What's my home Wi-Fi password?", "correct-horse", 20),
 ("Do I have any siblings?", "avery", 25),
 ("What's my favorite book?", "godel", 30),
 ("Where did I propose to my partner?", "machu picchu", 35),
 ("What car do I drive?", "subaru", 40),
 ("Have I ever donated an organ?", "kidney", 45),
]

print(f"Recall questions: {len(RECALL_QUESTIONS)}")

We run all 50 turns through the vector store memory. Each turn gets embedded and stored. After this, the memory contains 50 vectors ready for retrieval.

In [ ]:
# -- Run the 50-turn conversation through Vector Store Memory --

vec_mem = VectorStoreMemory(
 top_k=5,
 recent_buffer_size=4,
 system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
 collection_name="experiment_vec",
)

print("Running 50-turn conversation through Vector Store Memory...")
for turn_num, msg in conversation_50:
 vec_mem.chat(msg)
 if turn_num % 10 == 0:
 print(f" Turn {turn_num}/50 done")

print(f"\n\u2713 Done. Vectors stored: {vec_mem.collection.count()}")

Now we test recall. For each of the 12 planted facts, we ask a question and check whether the answer contains the expected keyword. We also log which turns the vector store retrieved, so we can see if it found the right source.

In [ ]:
# -- Recall test: Vector Store Memory --

print("=== Vector Store Memory \u2014 Recall Test ===\n")
vec_results = []

for question, keyword, planted_at in RECALL_QUESTIONS:
 # Check what would be retrieved
 retrieved = vec_mem.search(question, top_k=5)
 retrieved_turns = [r["turn"] for r in retrieved]

 answer = vec_mem.chat(question)
 recalled = keyword.lower() in answer.lower()
 vec_results.append({
 "question": question,
 "keyword": keyword,
 "planted_at": planted_at,
 "recalled": recalled,
 "retrieved_turns": retrieved_turns,
 "answer": answer,
 })

 status = "\u2713" if recalled else "\u2717"
 print(f" {status} (planted turn {planted_at:2d}) {question}")
 print(f" Retrieved turns: {retrieved_turns}")
 print(f" Answer: {answer[:100]}")
 print()

vec_score = sum(1 for r in vec_results if r["recalled"])
print(f"Vector Store Memory score: {vec_score}/{len(RECALL_QUESTIONS)}")

### Baseline: Sliding Window Memory (K=10)

For comparison, we run the same 50-turn conversation through a sliding window that keeps only the last 10 messages (5 user-assistant exchanges). This is a fair comparison: both systems use roughly the same amount of context per LLM call.

In [ ]:
class SlidingWindowMemory:
 """Simple sliding window: keep only the last K messages."""

 def __init__(self, window_size: int = 10, model: str = "claude-sonnet-4-20250514",
 system_prompt: str | None = None, max_tokens: int = 1024):
 self.window_size = window_size
 self.client = anthropic.Anthropic()
 self.model = model
 self.system_prompt = system_prompt
 self.max_tokens = max_tokens
 self.messages: list[dict] = []

 def chat(self, user_input: str) -> str:
 self.messages.append({"role": "user", "content": user_input})

 # Keep only the last window_size messages
 window = self.messages[-self.window_size:]

 kwargs = dict(model=self.model, max_tokens=self.max_tokens, messages=window)
 if self.system_prompt:
 kwargs["system"] = self.system_prompt

 response = self.client.messages.create(**kwargs)
 assistant_text = response.content[0].text

 self.messages.append({"role": "assistant", "content": assistant_text})
 return assistant_text


print("\u2713 SlidingWindowMemory class defined")

We run the same 50-turn conversation through the sliding window baseline. The window keeps only the last 10 messages (5 exchanges), so older facts fall out of context.

In [ ]:
# -- Run the same 50-turn conversation through Sliding Window --

sw_mem = SlidingWindowMemory(
 window_size=10,
 system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
)

print("Running 50-turn conversation through Sliding Window Memory...")
for turn_num, msg in conversation_50:
 sw_mem.chat(msg)
 if turn_num % 10 == 0:
 print(f" Turn {turn_num}/50 done")

print(f"\n\u2713 Done. Window keeps last {sw_mem.window_size} messages.")

We run the same recall questions against the sliding window. Since it only sees the last 10 messages, it should struggle with facts from early turns that have long since left the window.

In [ ]:
# -- Recall test: Sliding Window Memory --

print("=== Sliding Window Memory \u2014 Recall Test ===\n")
sw_results = []

for question, keyword, planted_at in RECALL_QUESTIONS:
 answer = sw_mem.chat(question)
 recalled = keyword.lower() in answer.lower()
 sw_results.append({
 "question": question,
 "keyword": keyword,
 "planted_at": planted_at,
 "recalled": recalled,
 "answer": answer,
 })

 status = "\u2713" if recalled else "\u2717"
 print(f" {status} (planted turn {planted_at:2d}) {question}")
 print(f" Answer: {answer[:100]}")
 print()

sw_score = sum(1 for r in sw_results if r["recalled"])
print(f"Sliding Window Memory score: {sw_score}/{len(RECALL_QUESTIONS)}")

Let's visualize the results. Panel 1 shows total recall scores. Panel 2 breaks down recall by the age of each fact (which turn it was planted at).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# -- Panel 1: Side-by-side recall scores --
labels = ["Vector Store\nMemory", "Sliding Window\n(K=10)"]
scores = [
 sum(1 for r in vec_results if r["recalled"]),
 sum(1 for r in sw_results if r["recalled"]),
]
colors = ["#4f46e5", "#ef4444"]
bars = axes[0].bar(labels, scores, color=colors, width=0.5, alpha=0.85)
axes[0].set_ylabel("Facts Recalled")
axes[0].set_ylim(0, len(RECALL_QUESTIONS) + 1)
axes[0].set_title("Total Facts Recalled (out of 12)")
axes[0].axhline(y=len(RECALL_QUESTIONS), color="gray", linestyle="--", alpha=0.3)
for bar, score in zip(bars, scores):
 axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
 str(score), ha="center", fontweight="bold", fontsize=14)

# -- Panel 2: Recall by turn distance (how old was the fact?) --
fact_turns = [r["planted_at"] for r in vec_results]
vec_recalled = [1 if r["recalled"] else 0 for r in vec_results]
sw_recalled = [1 if r["recalled"] else 0 for r in sw_results]

x = np.arange(len(fact_turns))
width = 0.35
axes[1].bar(x - width/2, vec_recalled, width, label="Vector Store", color="#4f46e5", alpha=0.85)
axes[1].bar(x + width/2, sw_recalled, width, label="Sliding Window", color="#ef4444", alpha=0.85)
axes[1].set_xlabel("Fact Planted at Turn #")
axes[1].set_ylabel("Recalled? (1=Yes, 0=No)")
axes[1].set_title("Recall by Fact Age")
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(t) for t in fact_turns], fontsize=8)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["No", "Yes"])
axes[1].legend()



The third panel checks retrieval precision. For each recall question, did the vector store retrieve the turn where the fact was originally planted? A green dot means yes. An orange X means the correct turn was missing from the top-K results.

In [ ]:
# -- Panel 3: What turns did the vector store retrieve? --
retrieved_hits = []
retrieved_misses = []
for r in vec_results:
 planted = r["planted_at"]
 retrieved = r.get("retrieved_turns", [])
 if planted in retrieved:
 retrieved_hits.append(planted)
 else:
 retrieved_misses.append(planted)

axes[2].scatter(retrieved_hits, [1]*len(retrieved_hits), color="#22c55e",
 s=120, marker="o", label="Retrieved correct turn", zorder=3)
axes[2].scatter(retrieved_misses, [0]*len(retrieved_misses), color="#f59e0b",
 s=120, marker="x", label="Missed correct turn", zorder=3)
axes[2].set_xlabel("Fact Planted at Turn #")
axes[2].set_ylabel("Correct Turn Retrieved?")
axes[2].set_title("Vector Store: Did It Retrieve the Right Turn?")
axes[2].set_yticks([0, 1])
axes[2].set_yticklabels(["No", "Yes"])
axes[2].legend(loc="center right")

plt.tight_layout()
plt.savefig("vector_vs_window.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nVector Store Memory: {scores[0]}/{len(RECALL_QUESTIONS)} recalled")
print(f"Sliding Window Memory: {scores[1]}/{len(RECALL_QUESTIONS)} recalled")

### Retrieval Quality Analysis

Let's look deeper at *what* the vector store retrieved for each recall question. Even when the answer is correct, it's informative to see which turns were surfaced and whether the "correct" turn was among them.

In [ ]:
print("=== Retrieval Analysis ===\n")
print(f"{'Question':<40} {'Planted':>7} {'Retrieved':>25} {'Hit?':>5}")
print("\u2500" * 85)

hits = 0
for r in vec_results:
 planted = r["planted_at"]
 retrieved = r.get("retrieved_turns", [])
 hit = planted in retrieved
 if hit:
 hits += 1
 print(f"{r['question'][:38]:<40} Turn {planted:<3} {str(retrieved):<25} {'\u2713' if hit else '\u2717':>5}")

print(f"\nRetrieval precision: {hits}/{len(vec_results)} queries retrieved the correct source turn.")

## Tuning Top-K: Recall vs. Token Cost

Choosing the right *K* is a classic precision-recall tradeoff:
- **Low K** (1-2): fewer tokens in the prompt, but you might miss relevant context.
- **High K** (10+): more likely to include the right memory, but adds token cost and may include irrelevant noise.

Let's measure how retrieval hit rate changes with K, using the same 50-turn conversation.

In [ ]:
# Test different K values against the stored vectors
k_values = [1, 2, 3, 5, 8, 10, 15]
k_hit_rates = []

for k in k_values:
 hits = 0
 for question, keyword, planted_at in RECALL_QUESTIONS:
 retrieved = vec_mem.search(question, top_k=k)
 retrieved_turns = [r["turn"] for r in retrieved]
 if planted_at in retrieved_turns:
 hits += 1
 hit_rate = hits / len(RECALL_QUESTIONS)
 k_hit_rates.append(hit_rate)
 print(f" K={k:2d}: {hits}/{len(RECALL_QUESTIONS)} hits ({hit_rate:.0%})")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_values, k_hit_rates, "o-", color="#4f46e5", linewidth=2, markersize=8)
ax.fill_between(k_values, k_hit_rates, alpha=0.1, color="#4f46e5")
ax.set_xlabel("Top-K")
ax.set_ylabel("Retrieval Hit Rate")
ax.set_title("Retrieval Hit Rate vs. Top-K")
ax.set_ylim(0, 1.05)
ax.set_xticks(k_values)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("topk_tuning.png", dpi=150, bbox_inches="tight")
plt.show()

We compare how token costs grow across three memory strategies. The full buffer sends every message on every call, so cost grows linearly. The sliding window is flat but forgets old context. The vector store stays flat while retaining access to all past turns.

In [ ]:
# Compare theoretical token costs across approaches

MSG_TOKENS = 50 # avg tokens per message
SYS_TOKENS = 30 # system prompt
K_RETRIEVE = 5 # top-K retrieved memories
NUM_TURNS = 50

full_buffer, sliding_window, vector_store = [], [], []

for turn in range(1, NUM_TURNS + 1):
 n_msgs = turn * 2

 # Full buffer: all messages
 full_buffer.append(SYS_TOKENS + n_msgs * MSG_TOKENS)

 # Sliding window (K=10 messages)
 sliding_window.append(SYS_TOKENS + min(n_msgs, 10) * MSG_TOKENS)

 # Vector store: K retrieved memories + small recent buffer (4 msgs)
 retrieved_tokens = K_RETRIEVE * 2 * MSG_TOKENS # each memory has user+assistant
 recent_tokens = min(n_msgs, 4) * MSG_TOKENS
 vector_store.append(SYS_TOKENS + retrieved_tokens + recent_tokens)

turns = list(range(1, NUM_TURNS + 1))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(turns, full_buffer, "o-", color="#ef4444", label="Full Buffer", linewidth=2, markersize=3)
ax.plot(turns, sliding_window, "s-", color="#f59e0b", label="Sliding Window (K=10)", linewidth=2, markersize=3)
ax.plot(turns, vector_store, "^-", color="#4f46e5", label="Vector Store (top-5 + buffer-4)", linewidth=2, markersize=3)

ax.set_xlabel("Conversation Turn")
ax.set_ylabel("Input Tokens per LLM Call")
ax.set_title("Token Cost per Turn: Three Memory Strategies (50 turns)")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("token_cost_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"At turn 50:")
print(f" Full buffer: {full_buffer[-1]:,} tokens/call")
print(f" Sliding window: {sliding_window[-1]:,} tokens/call")
print(f" Vector store: {vector_store[-1]:,} tokens/call")
print(f"\nVector store keeps cost constant while maintaining full recall.")

## Discussion & Tradeoffs

### Strengths
- **Semantic recall**: Retrieves context by *meaning*, not position. A fact from turn 3 is as accessible at turn 50 as at turn 4.
- **Bounded token cost**: Per-turn cost is roughly constant: `top_K x chunk_size + recent_buffer`. It stays flat regardless of conversation length.
- **Scalable**: Vector databases handle millions of vectors. Your memory can grow without degrading retrieval speed (with ANN indexing).
- **Composable**: Works naturally with metadata filters (time ranges, topics, user IDs) for more targeted retrieval.

### Weaknesses
- **Infrastructure overhead**: Requires an embedding model and a vector database. This adds complexity, latency, and cost per turn.
- **Embedding quality bottleneck**: If the embedding model poorly captures your domain's semantics, retrieval will be unreliable.
- **No narrative continuity**: Retrieved memories are isolated fragments, not a continuous story. The LLM must reconstruct coherence from scattered pieces.
- **Semantic does not equal relevant**: Two passages can be semantically similar but contextually irrelevant (e.g., "I like Python" and "Python is a snake").
- **Cold start**: With few stored memories, retrieval is noisy. The system improves as the memory store grows.

### When to Use Vector Store Memory

| Scenario | Recommendation |
|----------|---------------|
| Long conversations (50+ turns) where users revisit earlier topics | Good. Ideal use case. |
| Multi-session agents that must recall prior conversations | Good. Embed across sessions. |
| Real-time chat where every millisecond matters | Caution. Embedding adds ~50-100ms latency. |
| Short conversations (< 10 turns) | Overkill. Sliding window is sufficient. |
| Need for exact chronological narrative | Not ideal. Use buffer or summary instead. |
| Hybrid: recall old facts + understand recent flow | Good. Combine vector store with a recent buffer (as we did here). |

### Cost Model
Per turn:
- **1 embedding call** for the user query (~$0.00002 with `text-embedding-3-small`)
- **1 embedding call** for storing the exchange (~$0.00002)
- **1 vector search** (free with ChromaDB in-memory, ~10ms)
- **1 LLM call** with `top_K x chunk_size + buffer_size` tokens

The embedding cost is negligible compared to the LLM call. The real savings come from *not* sending the entire conversation history.

## Further Reading

- [ChromaDB Documentation](https://docs.trychroma.com/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): The in-memory vector database used in this notebook
- [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): How text-embedding-3-small works and pricing
- [LangChain VectorStoreRetrieverMemory](https://python.langchain.com/docs/modules/memory/types/vectorstore_retriever_memory?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Framework-level integration
- [Pinecone: Vector Database for Production](https://www.pinecone.io/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Managed vector DB alternative
- [FAISS: Facebook AI Similarity Search](https://github.com/facebookresearch/faiss): High-performance local vector search
- [Anthropic: Building Conversational AI](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Multi-turn conversation patterns
- [Lilian Weng, "LLM Powered Autonomous Agents"](https://lilianweng.github.io/posts/2023-06-23-agent/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Memory section covers vector retrieval patterns

---

*← Previous: [05: Token Buffer Memory](../05_token_buffer_memory/) · Next: [07: Entity Memory](../07_entity_memory/) →*

In [ ]:
# Clean up temp files created during the demo
import os
for f in ["vector_vs_window.png", "topk_tuning.png", "token_cost_comparison.png"]:
 if os.path.exists(f):
 os.remove(f)
 print(f"Removed {f}")

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Embedding model comparison
Swap `text-embedding-3-small` for `text-embedding-3-large` in `_embed()`. Run the same 50-turn experiment and compare hit rates at K=5. Record the embedding latency for each model to see the speed-quality tradeoff.

### Challenge 2: K parameter sweep
Run retrieval with K values of 1, 3, 5, 10, and 20. For each K, measure the fact retention rate using the recall evaluation. Plot K vs. recall and K vs. average input tokens per turn. Find the K that maximizes recall per token spent.

### Challenge 3: Metadata-filtered retrieval
Add a `turn_number` metadata field to each stored exchange. Implement a `_retrieve()` variant that filters by recency (only consider the last N turns for vector search). Compare this time-bounded vector search against pure semantic retrieval, drawing on ideas from 18 Temporal Memory.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--06-vector-store-memory--vector-store-memory)
